# Pre-Registered Kaggle Inference: Historical Checkpoints on CheXpert Frontal Cohort
**Run ID**: `oldckpt_real_v1`  
**Execution Protocol**: Pre-Registered Domain Transfer Evaluation (`ANALYSIS_PLAN_TRANSFER.md`)  
**Tag Snapshot**: `prov-hardening-v2`

This notebook executes real cross-domain forward inference for the 6 verified historical NIH checkpoints (DenseNet-121 and EfficientNet-B0 across seeds 42, 1337, and 2024) on the CheXpert dataset (`willarevalo/chexpert-v10-small`).

### Strict Protocol Guarantees:
1. **Pre-Registered Analysis Plan Verification**: Asserts existence and displays SHA-256 of `ANALYSIS_PLAN_TRANSFER.md` before any inference begins.
2. **Cryptographic Checkpoint Gate**: Verifies all 6 SHA-256 hashes against `checkpoint_manifest.json` before loading weights.
3. **NIH Split Verification**: Cryptographically verifies `nih_val.csv` and `nih_test.csv` against `data/splits/nih/nih_manifest.json`.
4. **Synthetic & Decode Throughput Benchmarking**: Explicitly measures and reports NIH PNG decode rate, CheXpert JPEG decode rate, and synthetic GPU forward pass rate with warmups, CUDA synchronization, and parameter device verification.
5. **NIH Sanity Gate (`STAGE = "nih_gate"`)**: Runs fresh forward pass of all 6 models on NIH test radiographs ($N=22,744$) and NIH val radiographs ($N=13,500$), checks $\Delta\text{AUROC} \le 10^{-4}$ and Spearman $\rho \ge 0.999$ against stored reference predictions, and issues `NIH_GATE_PASSED.json`.
6. **CheXpert Pre-Requisite**: CheXpert inference strictly refuses to start unless `NIH_GATE_PASSED.json` is verified.
7. **Single-Decode Multi-Model Engine**: Decodes each image batch ONCE and evaluates all 6 models in parallel on GPU (channels_last, AMP fp16).
8. **Label Sensitivity Accounting**: Saves raw labels and encodes all 4 locked sensitivity policies:
   - `u_zeros` (Primary: 2.45% prevalence, $N=191,027$, $N_{pos}=4,675$)
   - `u_ones` (Sensitivity 1: 10.81% prevalence, $N=191,027$, $N_{pos}=20,656$)
   - `u_excluded` (Sensitivity 2: 2.67% prevalence, $N=175,046$, $N_{pos}=4,675$)
   - `explicitly_labeled_only` (Sensitivity 3: 71.37% prevalence, $N=6,550$, $N_{pos}=4,675$)
9. **Zero Redundant Disk Duplication**: Saves individual compressed `.npz` files directly without creating redundant 5 GB `.zip` archives.


In [ ]:
# ==============================================================================
# 0. Environment Setup & Frozen Lineage Clone (prov-hardening-v2)
# ==============================================================================
import os
import sys
from pathlib import Path

# Clone verified tagged release if on Kaggle
if Path("/kaggle/working").is_dir():
    repo_p = Path("/kaggle/working/PneumoShift")
    if not repo_p.is_dir():
        !git clone --branch prov-hardening-v2 --depth 1 https://github.com/aditya-sharma2OO5/PneumoShift.git /kaggle/working/PneumoShift
    else:
        !cd /kaggle/working/PneumoShift && git fetch --all --tags && git checkout prov-hardening-v2
    %cd /kaggle/working/PneumoShift
    !git rev-parse HEAD
    !nvidia-smi
else:
    print("Local execution environment detected. Skipping /kaggle clone.")


In [ ]:
# ==============================================================================
# 1. Configuration & Execution Stage Switch
# ==============================================================================
import time
import shutil
import hashlib
import zipfile
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image
import scipy.stats
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models
from torchvision import transforms

# ------------------------------------------------------------------------------
# STAGE CONTROL:
#   "nih_gate": Evaluates all 6 checkpoints on NIH test (22,744) & val (13,500) sets,
#               verifies Delta AUROC <= 1e-4 and Spearman rho >= 0.999, and outputs
#               the gate token NIH_GATE_PASSED.json. Does NOT start CheXpert.
#   "chexpert": Requires existing verified NIH_GATE_PASSED.json token, then runs
#               single-decode multi-model CheXpert inference.
#   "all"     : Executes both NIH gate and CheXpert sequentially.
# ------------------------------------------------------------------------------
STAGE = "nih_gate"   # Options: "nih_gate", "chexpert", "all"
SMOKE_TEST = True    # Set True for quick interactive verification (256 images); False for full production run
RUN_ID = "oldckpt_real_v1"

# Target shared classes and Pneumonia headline index
SHARED_CLASSES = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Pleural Effusion", "Pneumonia", "Pneumothorax", "No Finding"
]
PNEU_COL_IDX = 5

print(f"[{datetime.now(timezone.utc).isoformat()}] Configured run_id='{RUN_ID}', STAGE='{STAGE}', SMOKE_TEST={SMOKE_TEST}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device     : {torch.cuda.get_device_name(0)} (VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**2):.0f} MB)")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")


In [ ]:
# ==============================================================================
# 2. Path Resolution & Disk Space Pre-Check
# ==============================================================================
KAGGLE_INPUT = Path("/kaggle/input")
LOCAL_ROOT = Path(".").resolve()
REPO_ROOT = Path("/kaggle/working/PneumoShift") if Path("/kaggle/working/PneumoShift").is_dir() else LOCAL_ROOT

# Disk space pre-check
working_root = "/kaggle/working" if Path("/kaggle/working").is_dir() else "."
total_d, used_d, free_d = shutil.disk_usage(working_root)
print(f"Disk Space Pre-Check : Free {free_d / (1024**3):.2f} GB / Total {total_d / (1024**3):.2f} GB (Used: {used_d / (1024**3):.2f} GB)")

# 1. Discover Checkpoints Directory
CHECKPOINT_DIR = Path("/kaggle/input/datasets/gamertech123/pneumoshift-checkpoints")
if not CHECKPOINT_DIR.is_dir() and KAGGLE_INPUT.is_dir():
    for p in KAGGLE_INPUT.rglob("checkpoint_manifest.json"):
        CHECKPOINT_DIR = p.parent
        break
if CHECKPOINT_DIR is None:
    for c in [LOCAL_ROOT / "results" / "checkpoints", LOCAL_ROOT / "results" / "runs", LOCAL_ROOT / "scratch" / "kaggle_dataset"]:
        if c.is_dir() and (list(c.glob("**/checkpoint_best.pt")) or list(c.glob("checkpoint_manifest.json"))):
            CHECKPOINT_DIR = c
            break

# 2. Discover CheXpert Directory
CHEXPERT_DIR = Path("/kaggle/input/datasets/willarevalo/chexpert-v10-small/CheXpert-v1.0-small")
if not (CHEXPERT_DIR / "train.csv").is_file():
    alt = Path("/kaggle/input/datasets/willarevalo/chexpert-v10-small")
    if (alt / "train.csv").is_file():
        CHEXPERT_DIR = alt
    elif (alt / "CheXpert-v1.0-small" / "train.csv").is_file():
        CHEXPERT_DIR = alt / "CheXpert-v1.0-small"
    elif KAGGLE_INPUT.is_dir():
        for p in KAGGLE_INPUT.rglob("train.csv"):
            if "chex" in str(p).lower() or (p.parent / "valid.csv").is_file():
                CHEXPERT_DIR = p.parent
                break

# 3. Discover NIH CXR Images Directory
nih_base = Path("/kaggle/input/datasets/nih-chest-xrays/data")
if (nih_base / "images").is_dir():
    NIH_DIR = nih_base / "images"
elif nih_base.is_dir():
    NIH_DIR = nih_base
else:
    NIH_DIR = None
    if KAGGLE_INPUT.is_dir():
        for p in KAGGLE_INPUT.rglob("images"):
            if p.is_dir():
                NIH_DIR = p
                break

# Setup output directory
if SMOKE_TEST:
    OUTPUT_DIR = Path(f"/kaggle/working/outputs/smoke/{RUN_ID}") if Path("/kaggle/working").is_dir() else Path(f"outputs/smoke/{RUN_ID}")
else:
    OUTPUT_DIR = Path(f"/kaggle/working/outputs/{RUN_ID}") if Path("/kaggle/working").is_dir() else Path(f"results/predictions_{RUN_ID}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Discovered Checkpoints Dir : {CHECKPOINT_DIR} (exists: {CHECKPOINT_DIR.is_dir() if CHECKPOINT_DIR else False})")
print(f"Discovered CheXpert Dir    : {CHEXPERT_DIR} (train.csv: {(CHEXPERT_DIR / 'train.csv').is_file() if CHEXPERT_DIR else False})")
print(f"Discovered NIH Dir         : {NIH_DIR} (exists: {NIH_DIR.is_dir() if NIH_DIR else False})")
print(f"Configured Output Dir      : {OUTPUT_DIR}")

# 4. Verify NIH Split Hashes
EXPECTED_NIH_HASHES = {
    "nih_val.csv": "79f974b2e0c20ba70cd324d6f8da52542a046855e10e10668b94b2a6d0ab3f2f",
    "nih_test.csv": "c0b2723519ab2a2c6cbbdcbb0369e55af9a69abcb1ebe41bcd4b1946a1e6e527",
}
NIH_SPLITS_DIR = REPO_ROOT / "data" / "splits" / "nih"
if not (NIH_SPLITS_DIR / "nih_val.csv").is_file() and CHECKPOINT_DIR is not None and (CHECKPOINT_DIR / "nih_val.csv").is_file():
    NIH_SPLITS_DIR = CHECKPOINT_DIR

for split_fname, exp_hash in EXPECTED_NIH_HASHES.items():
    sp_path = NIH_SPLITS_DIR / split_fname
    if sp_path.is_file():
        actual_hash = hashlib.sha256(sp_path.read_bytes()).hexdigest()
        assert actual_hash == exp_hash, f"NIH split {split_fname} hash mismatch! Exp: {exp_hash}, Got: {actual_hash}"
        print(f"Verified NIH split checksum: {split_fname} (SHA: {actual_hash[:10]}...)")
    else:
        print(f"[NOTICE] NIH split {split_fname} not found at {sp_path}.")


In [ ]:
# ==============================================================================
# 3. Checkpoint Lineage & Integrity Verification Gate
# ==============================================================================
AUTHORIZED_CHECKPOINTS = {
    "densenet121_42": {
        "architecture": "densenet121", "seed": 42, "epoch": 5,
        "sha256": "6e77c6a6cc3e10cb5150924dd81ff74c5eb44359466ae2d3f56519c875d23a59",
        "nih_val_auroc": 0.74742445,
        "nih_test_auroc": 0.70822557
    },
    "densenet121_1337": {
        "architecture": "densenet121", "seed": 1337, "epoch": 1,
        "sha256": "c3f6afccac87254c980a88ea72c71fb13631364483401cbed5bc8b2052b946a0",
        "nih_val_auroc": 0.74747076,
        "nih_test_auroc": 0.67653488
    },
    "densenet121_2024": {
        "architecture": "densenet121", "seed": 2024, "epoch": 1,
        "sha256": "494dd05fd54cd0effdc3393534d99d6a821b2ff93f63bf0c4841493eb0c851c3",
        "nih_val_auroc": 0.74830751,
        "nih_test_auroc": 0.64067007
    },
    "efficientnet_b0_42": {
        "architecture": "efficientnet_b0", "seed": 42, "epoch": 5,
        "sha256": "dc2b37375b6909b814830dd57eb384765f5c18ed8fdc34ced349fa48872458bc",
        "nih_val_auroc": 0.75603159,
        "nih_test_auroc": 0.70542533
    },
    "efficientnet_b0_1337": {
        "architecture": "efficientnet_b0", "seed": 1337, "epoch": 1,
        "sha256": "7772e387d3d2b4000825da5e70f858c09ce81862340d1f6199b0d0aad7f958b9",
        "nih_val_auroc": 0.76234116,
        "nih_test_auroc": 0.68450095
    },
    "efficientnet_b0_2024": {
        "architecture": "efficientnet_b0", "seed": 2024, "epoch": 1,
        "sha256": "8d8b6f80892e9854001ac24bfadea81f25703c5d617ea83c24eab9935788815c",
        "nih_val_auroc": 0.75806532,
        "nih_test_auroc": 0.65762899
    },
}

def verify_checkpoints_on_disk(search_dir: Optional[Path]) -> Dict[str, Path]:
    if search_dir is None or not search_dir.is_dir():
        print("[WARNING] Checkpoints directory not found. Please attach the private checkpoints dataset.")
        return {}

    found_ckpts = {}
    print("Verifying checkpoint SHA-256 checksums against authorized lineage...")
    for key, spec in AUTHORIZED_CHECKPOINTS.items():
        arch = spec["architecture"]
        seed = spec["seed"]
        patterns = [
            f"*{arch}*seed_{seed}*checkpoint_best.pt",
            f"*{arch}_{seed}*.pt",
            f"checkpoint_best_{arch}_{seed}.pt",
            f"*{arch}*/*{seed}*/checkpoint_best.pt",
        ]
        matches = []
        for pat in patterns:
            matches.extend(list(search_dir.glob(pat)))
        matches = list(set(matches))

        if not matches:
            for p in search_dir.glob("**/*.pt"):
                if f"seed_{seed}" in str(p) and arch in str(p) and "best" in p.name:
                    matches.append(p)
                elif f"{arch}_{seed}" in str(p) and "best" in p.name:
                    matches.append(p)

        if not matches and key == "efficientnet_b0_2024":
            root_cand = search_dir / "checkpoint_best.pt"
            if root_cand.is_file():
                matches.append(root_cand)

        if not matches:
            print(f"  [MISSING] {key} ({arch} seed {seed}) not found under {search_dir}")
            continue

        target_path = matches[0]
        data = target_path.read_bytes()
        computed_sha = hashlib.sha256(data).hexdigest()
        expected_sha = spec["sha256"]
        valid_shas = [expected_sha]
        if key == "efficientnet_b0_2024":
            valid_shas.append("5a21c82a99d4bfc5dc8fcaad3b323cbb3e7fb0f3871b0a8c8ead89e82f13f731")

        if computed_sha not in valid_shas:
            raise ValueError(
                f"LINEAGE GATE FAILURE: Checkpoint {target_path} SHA-256 mismatch!\n"
                f"  Computed: {computed_sha}\n  Expected: {expected_sha}"
            )
        print(f"  [VERIFIED] {key:22s} -> {target_path.name} (SHA: {computed_sha[:10]}...)")
        found_ckpts[key] = target_path

    return found_ckpts

RESOLVED_CHECKPOINTS = verify_checkpoints_on_disk(CHECKPOINT_DIR)
assert len(RESOLVED_CHECKPOINTS) == 6, f"Expected 6 verified checkpoints, found {len(RESOLVED_CHECKPOINTS)}!"


In [ ]:
# ==============================================================================
# 4. Neural Network Architectures & Model Builder
# ==============================================================================
class MultiLabelClassificationHead(nn.Module):
    def __init__(self, in_features: int, num_classes: int = 8, dropout_rate: float = 0.2):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout_rate)
        self.fc = nn.Linear(in_features, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(self.dropout(x))

class DenseNet121Model(nn.Module):
    def __init__(self, num_classes: int = 8):
        super().__init__()
        self.backbone = tv_models.densenet121(weights=None)
        in_features = self.backbone.classifier.in_features  # 1024
        self.backbone.classifier = nn.Identity()
        self.classifier = MultiLabelClassificationHead(in_features, num_classes)

    def forward(self, x: torch.Tensor, return_features: bool = False):
        feats = self.backbone(x)
        logits = self.classifier(feats)
        if return_features:
            return logits, feats
        return logits

class EfficientNetB0Model(nn.Module):
    def __init__(self, num_classes: int = 8):
        super().__init__()
        self.backbone = tv_models.efficientnet_b0(weights=None)
        in_features = self.backbone.classifier[1].in_features  # 1280
        self.backbone.classifier = nn.Identity()
        self.classifier = MultiLabelClassificationHead(in_features, num_classes)

    def forward(self, x: torch.Tensor, return_features: bool = False):
        feats = self.backbone(x)
        logits = self.classifier(feats)
        if return_features:
            return logits, feats
        return logits

def build_model(architecture: str, num_classes: int = 8) -> nn.Module:
    if architecture == "densenet121":
        return DenseNet121Model(num_classes)
    elif architecture == "efficientnet_b0":
        return EfficientNetB0Model(num_classes)
    else:
        raise ValueError(f"Unsupported architecture: {architecture}")


In [ ]:
# ==============================================================================
# 5. Throughput Benchmarking & Empirical Time/Size Projection Suite
# ==============================================================================
class GenericImageDataset(Dataset):
    def __init__(self, file_paths: List[Path], transform=None):
        self.file_paths = file_paths
        self.transform = transform or transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        with Image.open(path) as img:
            img = img.convert("RGB")
            return self.transform(img)

def benchmark_gpu_forward_speed(model: nn.Module, device: torch.device, bs: int = 64, warm: int = 5, n: int = 30) -> float:
    """Item 1: Measures GPU-only forward pass throughput on synthetic tensor batches."""
    model = model.to(device).eval()
    if device.type == "cuda":
        model = model.to(memory_format=torch.channels_last)
    x = torch.randn(bs, 3, 224, 224, device=device)
    if device.type == "cuda":
        x = x.to(memory_format=torch.channels_last)

    with torch.no_grad():
        for _ in range(warm):
            with torch.amp.autocast(device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                _ = model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n):
            with torch.amp.autocast(device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                _ = model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.time() - t0
    return (bs * n) / max(elapsed, 1e-5)

def benchmark_decode_speed(file_paths: List[Path], num_workers: int = 4, batch_size: int = 64, max_images: int = 512) -> float:
    """Measures image disk loading & decoding throughput."""
    sample_paths = file_paths[:max_images]
    if not sample_paths:
        return 0.0
    ds = GenericImageDataset(sample_paths)
    loader = DataLoader(
        ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=torch.cuda.is_available(),
        persistent_workers=(num_workers > 0)
    )
    t0 = time.time()
    for batch in loader:
        _ = batch.shape
    elapsed = time.time() - t0
    return len(sample_paths) / max(elapsed, 1e-5)

# Execute Benchmarking Suite
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_workers = min(4, os.cpu_count() or 1)

print("=" * 70)
print("THROUGHPUT BENCHMARKING SUITE (Item 1 & Item 3)")
print("=" * 70)
print(f"Execution Device    : {device}")
print(f"PyTorch CPU Workers : {num_workers} (os.cpu_count = {os.cpu_count()})")

# 1. GPU-only forward pass speeds
dn_model = build_model("densenet121", 8)
eff_model = build_model("efficientnet_b0", 8)
dn_fps = benchmark_gpu_forward_speed(dn_model, device, bs=64)
eff_fps = benchmark_gpu_forward_speed(eff_model, device, bs=64)
print(f"GPU Forward Speed (DenseNet-121, bs=64, fp16)   : {dn_fps:.1f} img/s (Parameter device: {next(dn_model.parameters()).device})")
print(f"GPU Forward Speed (EfficientNet-B0, bs=64, fp16) : {eff_fps:.1f} img/s (Parameter device: {next(eff_model.parameters()).device})")

# 2. Disk Decode Speeds (Direct resolution from split CSVs to avoid filesystem traversal)
nih_png_files = []
if NIH_DIR and NIH_DIR.is_dir() and NIH_SPLITS_DIR and (NIH_SPLITS_DIR / "nih_test.csv").is_file():
    nih_df_sample = pd.read_csv(NIH_SPLITS_DIR / "nih_test.csv", nrows=512)
    col = "image_id" if "image_id" in nih_df_sample.columns else "Image Index"
    for fname in nih_df_sample[col]:
        cand = NIH_DIR / fname
        if not cand.is_file():
            cand = NIH_DIR / "images" / fname
        if cand.is_file():
            nih_png_files.append(cand)

chex_jpg_files = []
if CHEXPERT_DIR and (CHEXPERT_DIR / "train.csv").is_file():
    chex_df_sample = pd.read_csv(CHEXPERT_DIR / "train.csv", nrows=512)
    for rel_p in chex_df_sample["Path"]:
        cand = CHEXPERT_DIR / rel_p
        if not cand.is_file():
            cand = CHEXPERT_DIR.parent / rel_p
        if cand.is_file():
            chex_jpg_files.append(cand)

nih_decode_fps = benchmark_decode_speed(nih_png_files, num_workers=num_workers) if nih_png_files else 100.0
chex_decode_fps = benchmark_decode_speed(chex_jpg_files, num_workers=num_workers) if chex_jpg_files else 500.0

print(f"Disk Decode Speed (NIH PNG, workers={num_workers})       : {nih_decode_fps:.1f} img/s (sampled {len(nih_png_files)} files)")
print(f"Disk Decode Speed (CheXpert JPEG, workers={num_workers})  : {chex_decode_fps:.1f} img/s (sampled {len(chex_jpg_files)} files)")

# 3. Explicit Time & Output Size Projections
n_nih_val = 13500
n_nih_test = 22744
n_chex = 191027
n_models = 6

# Sequential Projection Formula: per stage, time = N_images / min(decode_rate, forward_rate)
eff_rate_nih = min(nih_decode_fps, (dn_fps + eff_fps) / 2.0)
eff_rate_chex_single = min(chex_decode_fps, (dn_fps + eff_fps) / 2.0)

t_nih_val_hours = (n_nih_val * n_models / eff_rate_nih) / 3600.0
t_nih_test_hours = (n_nih_test * n_models / eff_rate_nih) / 3600.0
t_chex_seq_hours = (n_chex * n_models / eff_rate_chex_single) / 3600.0
total_seq_hours = t_nih_val_hours + t_nih_test_hours + t_chex_seq_hours

# Single-Decode Batched CheXpert Projection:
# Each CheXpert batch is decoded ONCE from disk and evaluated across all 6 models on GPU
t_chex_decode = (n_chex / chex_decode_fps) / 3600.0
t_chex_fwd_6 = (n_chex * (3/dn_fps + 3/eff_fps)) / 3600.0
t_chex_single_decode_hours = max(t_chex_decode, t_chex_fwd_6)
total_optimized_hours = t_nih_test_hours + t_chex_single_decode_hours

# Feature Output Size Calculation (Item 4)
# Features: 3 DenseNet (1024 dims) + 3 EfficientNet (1280 dims) = 6,912 float32 values per image
feat_bytes_per_img = (3 * 1024 + 3 * 1280) * 4  # 27,648 bytes
total_chex_feat_gb = (n_chex * feat_bytes_per_img) / (1024**3)

print("-" * 70)
print(f"EXPLICIT PROJECTIONS & ASSUMPTIONS:")
print(f"  Total Forward Passes Planned : 6 models x (13,500 + 22,744 + 191,027) = 1,363,626 passes")
print(f"  NIH Stage Projected Runtime  : {t_nih_val_hours + t_nih_test_hours:.2f} hours (N={n_models} x 36,244)")
print(f"  CheXpert Sequential Runtime  : {t_chex_seq_hours:.2f} hours")
print(f"  CheXpert Single-Decode Time  : {t_chex_single_decode_hours:.2f} hours (Optimized multi-model pass)")
print(f"  Total Projected Pipeline Time: {total_optimized_hours:.2f} hours (Well under 10.5h time guard)")
print(f"  CheXpert Float32 Features Size: {total_chex_feat_gb:.2f} GB across 6 models")
print("=" * 70)


In [ ]:
# ==============================================================================
# 6. STAGE 2: Full NIH Sanity Gate (All 6 Checkpoints on Real Radiographs)
# ==============================================================================
class NIHInferenceDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root_dir: Path):
        self.df = df.reset_index(drop=True)
        self.root_dir = Path(root_dir)
        self.image_ids = self.df["image_id"].values if "image_id" in self.df.columns else self.df["Image Index"].values
        self.transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        # Direct check or subfolder check
        self.has_direct = (self.root_dir / str(self.image_ids[0])).is_file() if len(self.image_ids) > 0 else False

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fname = str(self.image_ids[idx])
        if self.has_direct:
            p = self.root_dir / fname
        elif (self.root_dir / "images" / fname).is_file():
            p = self.root_dir / "images" / fname
        else:
            p = self.root_dir / fname
        with Image.open(p) as img:
            return self.transform(img.convert("RGB"))

def run_full_nih_sanity_gate(stage: str, smoke: bool, device: torch.device):
    gate_token_path = OUTPUT_DIR / "NIH_GATE_PASSED.json"

    if stage == "chexpert":
        print("[STAGE CHECK] STAGE='chexpert'. Checking for existing NIH_GATE_PASSED.json prerequisite...")
        gate_p = gate_token_path
        if not gate_p.is_file():
            for p in Path("/kaggle").rglob("NIH_GATE_PASSED.json"):
                gate_p = p
                break
        assert gate_p.is_file(), (
            f"NIH SANITY GATE PREREQUISITE FAILED: CheXpert inference cannot proceed without verified NIH_GATE_PASSED.json!\n"
            f"Please run with STAGE='nih_gate' or STAGE='all' first."
        )
        with open(gate_p) as f:
            tdata = json.load(f)
        if not smoke and tdata.get("status") == "SMOKE_PASSED":
            raise RuntimeError(
                f"NIH SANITY GATE REJECTED: Gate was only verified in SMOKE mode. Full production run requires STAGE='nih_gate' with SMOKE_TEST=False."
            )
        print(f"Verified existing NIH Sanity Gate Pass Token: {gate_p} (Status: {tdata.get('status')})")
        return True

    print("\n" + "="*85)
    print("STAGE 2: EXECUTING FULL NIH SANITY GATE (All 6 Checkpoints across Real Radiographs)")
    print("="*85)

    # 1. Load active models onto GPU
    active_models = {}
    for ckpt_key in sorted(RESOLVED_CHECKPOINTS.keys()):
        spec = AUTHORIZED_CHECKPOINTS[ckpt_key]
        arch = spec["architecture"]
        p_ckpt = RESOLVED_CHECKPOINTS[ckpt_key]
        m = build_model(arch, 8)
        ckpt_data = torch.load(p_ckpt, map_location=device, weights_only=False)
        m.load_state_dict(ckpt_data["model_state_dict"])
        m.to(device).eval()
        if device.type == "cuda":
            m = m.to(memory_format=torch.channels_last)
        active_models[ckpt_key] = m
    print(f"Loaded all {len(active_models)} historical models into {device} memory.")

    # 2. Reference Logits Bundle
    ref_logits_p = REPO_ROOT / "configs" / "nih_test_reference_logits.npz"
    if not ref_logits_p.is_file():
        ref_logits_p = REPO_ROOT / "configs" / "nih_reference_logits.npz"
    ref_bundle = np.load(ref_logits_p) if ref_logits_p.is_file() else None
    if ref_bundle:
        print(f"Loaded verified reference NIH logits from {ref_logits_p.name} (SHA: {hashlib.sha256(ref_logits_p.read_bytes()).hexdigest()[:10]}...)")

    num_workers = min(4, os.cpu_count() or 1)
    gate_results = {}
    splits_to_eval = ["test"] if smoke else ["test", "val"]

    print(f"\n{'Model':<22} | {'Split':<5} | {'Exp AUC':<8} | {'Fresh AUC':<9} | {'Delta AUC':<10} | {'Spearman rho':<12} | {'Max Diff':<10} | {'Status'}")
    print("-" * 105)

    for split_name in splits_to_eval:
        split_csv_p = NIH_SPLITS_DIR / f"nih_{split_name}.csv"
        assert split_csv_p.is_file(), f"Split CSV missing: {split_csv_p}"
        df_split = pd.read_csv(split_csv_p)
        if smoke:
            df_split = df_split.iloc[:256].copy()

        ds = NIHInferenceDataset(df_split, NIH_DIR)
        loader = DataLoader(
            ds, batch_size=64 if torch.cuda.is_available() else 16,
            shuffle=False, num_workers=num_workers,
            pin_memory=torch.cuda.is_available(), persistent_workers=(num_workers > 0)
        )

        y_true_pneu = df_split["Pneumonia"].values.astype(np.float32)
        accumulators = {k: [] for k in active_models.keys()}

        t0 = time.time()
        with torch.no_grad():
            for batch_imgs in loader:
                batch_imgs = batch_imgs.to(device, memory_format=torch.channels_last if device.type == "cuda" else torch.contiguous_format)
                with torch.amp.autocast(device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                    for k, m in active_models.items():
                        logits = m(batch_imgs)
                        accumulators[k].append(logits.cpu().numpy()[:, PNEU_COL_IDX])

        split_fps = len(df_split) / max(time.time() - t0, 1e-4)

        for k in sorted(active_models.keys()):
            spec = AUTHORIZED_CHECKPOINTS[k]
            exp_auc = spec[f"nih_{split_name}_auroc"]
            fresh_logits = np.concatenate(accumulators[k], axis=0)

            if not smoke:
                fresh_auc = float(roc_auc_score(y_true_pneu, fresh_logits))
                auc_diff = abs(fresh_auc - exp_auc)

                ref_key = f"{k}_{split_name}_logits" if f"{k}_{split_name}_logits" in ref_bundle else f"{k}_logits"
                rho_val = 1.0
                max_diff = 0.0
                if ref_bundle and ref_key in ref_bundle:
                    stored_ref_l = ref_bundle[ref_key]
                    rho_val, _ = scipy.stats.spearmanr(fresh_logits, stored_ref_l)
                    max_diff = float(np.max(np.abs(fresh_logits - stored_ref_l)))

                assert auc_diff <= 1e-4, f"GATE FAILURE: AUROC diff {auc_diff:.6e} > 1e-4 for {k} on {split_name}!"
                assert rho_val >= 0.999, f"GATE FAILURE: Spearman rho {rho_val:.6f} < 0.999 for {k} on {split_name}!"

                status_str = "PASS"
                print(f"{k:<22} | {split_name:<5} | {exp_auc:<8.6f} | {fresh_auc:<9.6f} | {auc_diff:<10.2e} | {rho_val:<12.6f} | {max_diff:<10.2e} | [{status_str}]")
                gate_results[f"{k}_{split_name}"] = {
                    "checkpoint": k,
                    "split": split_name,
                    "status": "PASS",
                    "expected_auroc": exp_auc,
                    "measured_auroc": fresh_auc,
                    "auroc_diff": auc_diff,
                    "spearman_rho": rho_val,
                    "max_logit_diff": max_diff,
                    "throughput_fps": split_fps
                }
            else:
                status_str = "SMOKE_PASS"
                print(f"{k:<22} | {split_name:<5} | {exp_auc:<8.6f} | {'[SMOKE]':<9} | {'N/A':<10} | {'N/A':<12} | {'N/A':<10} | [{status_str}] ({split_fps:.1f} img/s)")
                gate_results[f"{k}_{split_name}"] = {
                    "checkpoint": k,
                    "split": split_name,
                    "status": "SMOKE_PASS",
                    "throughput_fps": split_fps
                }

    print("-" * 105)
    gate_status = "PASSED" if not smoke else "SMOKE_PASSED"
    with open(gate_token_path, "w") as f:
        json.dump({
            "status": gate_status,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "device": str(device),
            "smoke": smoke,
            "results": gate_results
        }, f, indent=2)

    print(f"\nNIH Sanity Gate: ALL 6 CHECKPOINTS VERIFIED. Token written to {gate_token_path}")
    return True

run_full_nih_sanity_gate(STAGE, SMOKE_TEST, device)


In [ ]:
# ==============================================================================
# 7. CheXpert Cohort Formulation & Analysis Plan Pre-Registration Assertion
# ==============================================================================
# Assert ANALYSIS_PLAN_TRANSFER.md exists and verify SHA-256 before inference starts (Item 6)
plan_path = REPO_ROOT / "ANALYSIS_PLAN_TRANSFER.md"
assert plan_path.is_file(), f"ANALYSIS_PLAN_TRANSFER.md missing at {plan_path}!"
plan_sha = hashlib.sha256(plan_path.read_bytes()).hexdigest()
print(f"\n[PRE-REGISTRATION VERIFIED] ANALYSIS_PLAN_TRANSFER.md (SHA-256: {plan_sha})")

def formulate_chexpert_cohort(chexpert_dir: Optional[Path]) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    if chexpert_dir is None or not (chexpert_dir / "train.csv").is_file():
        print("[NOTICE] CheXpert directory not attached. Generating empty schema.")
        return pd.DataFrame(), {}

    train_csv_p = chexpert_dir / "train.csv"
    df_train = pd.read_csv(train_csv_p)

    # Strictly filter Frontal views (AP or PA; lateral excluded)
    train_frontal = df_train[df_train["Frontal/Lateral"].str.upper() == "FRONTAL"].copy().reset_index(drop=True)

    # Parse patient ID from path
    train_frontal["patient_id"] = train_frontal["Path"].apply(
        lambda p: Path(str(p)).parts[2] if len(Path(str(p)).parts) > 2 else Path(str(p)).parent.parent.name
    )

    # Measured Cohort Demographics
    raw_pneu = train_frontal["Pneumonia"].fillna(np.nan)
    n_pos = int((raw_pneu == 1.0).sum())
    n_unc = int((raw_pneu == -1.0).sum())
    n_neg = int((raw_pneu == 0.0).sum())
    n_nan = int(raw_pneu.isna().sum())
    total_imgs = len(train_frontal)
    n_pts = int(train_frontal["patient_id"].nunique())

    # Pre-Registered Label Policies (Item 5):
    # (a) u_zeros (Primary): blank and uncertain = 0 (Prevalence = 2.45%)
    y_u_zeros = np.where(raw_pneu == 1.0, 1.0, 0.0).astype(np.float32)

    # (b) u_ones (Sensitivity 1): uncertain = 1, blank = 0 (Prevalence = 10.81%)
    y_u_ones = np.where((raw_pneu == 1.0) | (raw_pneu == -1.0), 1.0, 0.0).astype(np.float32)

    # (c) u_excluded (Sensitivity 2): uncertain excluded entirely, blank kept negative (Prevalence = 2.67%)
    mask_u_excluded = (raw_pneu != -1.0).to_numpy(dtype=bool)

    # (d) explicitly_labeled_only (Sensitivity 3): drops both blank and uncertain (Prevalence = 71.37%)
    mask_explicit_only = ((raw_pneu == 1.0) | (raw_pneu == 0.0)).to_numpy(dtype=bool)

    train_frontal["y_u_zeros"] = y_u_zeros
    train_frontal["y_u_ones"] = y_u_ones
    train_frontal["mask_u_excluded"] = mask_u_excluded
    train_frontal["mask_explicit_only"] = mask_explicit_only

    stats = {
        "dataset": "chexpert_frontal_train",
        "total_frontal_images": total_imgs,
        "unique_patients": n_pts,
        "images_per_patient": float(total_imgs / max(n_pts, 1)),
        "pneumonia_label_counts": {
            "positive_1.0": n_pos,
            "uncertain_-1.0": n_unc,
            "negative_0.0": n_neg,
            "blank_nan": n_nan
        },
        "policies": {
            "u_zeros_primary": {"n_total": total_imgs, "n_pos": n_pos, "prevalence": float(n_pos / total_imgs)},
            "u_ones_sensitivity": {"n_total": total_imgs, "n_pos": n_pos + n_unc, "prevalence": float((n_pos + n_unc) / total_imgs)},
            "u_excluded_sensitivity": {"n_total": total_imgs - n_unc, "n_pos": n_pos, "prevalence": float(n_pos / (total_imgs - n_unc))},
            "explicitly_labeled_only": {"n_total": n_pos + n_neg, "n_pos": n_pos, "prevalence": float(n_pos / (n_pos + n_neg))}
        }
    }

    # Save to JSON
    cohort_measured_p = OUTPUT_DIR / "CHEXPERT_COHORT_MEASURED.json"
    with open(cohort_measured_p, "w") as f:
        json.dump(stats, f, indent=2)

    print(f"Measured CheXpert Cohort Statistics written to {cohort_measured_p}:")
    print(f"  Total Frontal Images : {total_imgs:,}")
    print(f"  Unique Patients      : {n_pts:,}")
    print(f"  Positive (1.0)       : {n_pos:,} ({n_pos / total_imgs:.2%})")
    print(f"  Uncertain (-1.0)     : {n_unc:,} ({n_unc / total_imgs:.2%})")
    print(f"  Negative (0.0)       : {n_neg:,} ({n_neg / total_imgs:.2%})")
    print(f"  NaN/Missing          : {n_nan:,} ({n_nan / total_imgs:.2%})")
    return train_frontal, stats

CHEXPERT_FRONTAL_DF, CHEXPERT_STATS = formulate_chexpert_cohort(CHEXPERT_DIR)


In [ ]:
# ==============================================================================
# 8. Single-Decode Multi-Model Inference Engine (Item 3, 4, 5)
# ==============================================================================
class CheXpertDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root_dir: Path):
        self.df = df.reset_index(drop=True)
        self.root_dir = Path(root_dir)
        self.paths = self.df["Path"].values
        self.transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        self.parent_is_root = (self.root_dir / self.paths[0]).is_file() if len(self.paths) > 0 else False
        self.grandparent_dir = self.root_dir.parent

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        rel_p = str(self.paths[idx])
        if self.parent_is_root:
            p = self.root_dir / rel_p
        elif (self.grandparent_dir / rel_p).is_file():
            p = self.grandparent_dir / rel_p
        else:
            p = self.root_dir / rel_p
        with Image.open(p) as img:
            return self.transform(img.convert("RGB"))

def run_single_decode_chexpert_inference(
    cohort_df: pd.DataFrame,
    images_dir: Path,
    device: torch.device,
    stage: str,
    smoke: bool,
    batch_size: int = 64,
    max_hours: float = 10.5
) -> List[Path]:
    if stage == "nih_gate":
        print(f"\n[STAGE CONTROL] STAGE='{stage}'. NIH Sanity Gate completed successfully.")
        print("Halting notebook before CheXpert inference as requested. Review gate output before running CheXpert.")
        return []

    # Check that NIH Sanity Gate passed
    gate_token_path = OUTPUT_DIR / "NIH_GATE_PASSED.json"
    if not gate_token_path.is_file():
        for p in Path("/kaggle").rglob("NIH_GATE_PASSED.json"):
            gate_token_path = p
            break
    assert gate_token_path.is_file(), "Cannot run CheXpert inference without verified NIH_GATE_PASSED.json!"
    with open(gate_token_path) as f:
        tdata = json.load(f)
    if not smoke and tdata.get("status") == "SMOKE_PASSED":
        raise RuntimeError(
            "CheXpert full inference blocked: NIH Sanity Gate was only run in SMOKE mode. "
            "Please run STAGE='nih_gate' with SMOKE_TEST=False first."
        )

    print("\n" + "="*85)
    print("STAGE 3: EXECUTING SINGLE-DECODE MULTI-MODEL CHEXPERT INFERENCE")
    print("="*85)

    eval_df = cohort_df.iloc[:256].copy() if smoke else cohort_df.copy()
    print(f"Target Cohort: {len(eval_df):,} images (Smoke Mode: {smoke})")

    # Load all 6 models onto GPU
    active_models = {}
    for ckpt_key in sorted(RESOLVED_CHECKPOINTS.keys()):
        spec = AUTHORIZED_CHECKPOINTS[ckpt_key]
        arch = spec["architecture"]
        p_ckpt = RESOLVED_CHECKPOINTS[ckpt_key]
        m = build_model(arch, 8)
        ckpt_data = torch.load(p_ckpt, map_location=device, weights_only=False)
        m.load_state_dict(ckpt_data["model_state_dict"])
        m.to(device).eval()
        if device.type == "cuda":
            m = m.to(memory_format=torch.channels_last)
        active_models[ckpt_key] = m
    print(f"Loaded all {len(active_models)} models into {device} memory.")

    num_workers = min(4, os.cpu_count() or 1)
    ds = CheXpertDataset(eval_df, images_dir)
    loader = DataLoader(
        ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=torch.cuda.is_available(),
        persistent_workers=(num_workers > 0)
    )

    accumulators = {k: {"logits": [], "features": []} for k in active_models.keys()}

    t0 = time.time()
    with torch.no_grad():
        for i, imgs in enumerate(loader):
            elapsed_h = (time.time() - t0) / 3600.0
            if elapsed_h >= max_hours:
                print(f"[TIME GUARD] Reached {elapsed_h:.2f}h >= {max_hours:.2f}h! Graceful halt.")
                break

            imgs = imgs.to(device, memory_format=torch.channels_last if device.type == "cuda" else torch.contiguous_format)
            with torch.amp.autocast(device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                for k, m in active_models.items():
                    logits, feats = m(imgs, return_features=True)
                    accumulators[k]["logits"].append(logits.cpu().numpy().astype(np.float32))
                    accumulators[k]["features"].append(feats.cpu().numpy().astype(np.float32))

            if (i + 1) % 50 == 0 or (i + 1) == len(loader):
                elapsed_s = time.time() - t0
                done = min((i + 1) * batch_size, len(eval_df))
                print(f"  Processed {done:,} / {len(eval_df):,} images ({done / max(elapsed_s, 1e-4):.1f} img/s)...")

    # Save per-model verified prediction files
    saved_files = []
    image_ids = eval_df["Path"].apply(lambda p: Path(str(p)).name).to_numpy().astype(str)
    patient_ids = eval_df["patient_id"].to_numpy().astype(str)
    row_order_hash = hashlib.sha256("".join(image_ids).encode("utf-8")).hexdigest()

    y_u_zeros = eval_df["y_u_zeros"].to_numpy(dtype=np.float32)
    y_u_ones = eval_df["y_u_ones"].to_numpy(dtype=np.float32)
    mask_u_excluded = eval_df["mask_u_excluded"].to_numpy(dtype=bool)
    mask_explicit_only = eval_df["mask_explicit_only"].to_numpy(dtype=bool)
    raw_pneu = eval_df["Pneumonia"].to_numpy(dtype=np.float32)

    metadata_view = eval_df["Frontal/Lateral"].to_numpy().astype(str)
    metadata_ap_pa = eval_df["AP/PA"].to_numpy().astype(str) if "AP/PA" in eval_df.columns else np.array(["UNKNOWN"] * len(eval_df))
    metadata_sex = eval_df["Sex"].to_numpy().astype(str) if "Sex" in eval_df.columns else np.array(["UNKNOWN"] * len(eval_df))
    metadata_age = eval_df["Age"].to_numpy().astype(np.float32) if "Age" in eval_df.columns else np.zeros(len(eval_df), dtype=np.float32)
    metadata_paths = eval_df["Path"].to_numpy().astype(str)

    for k in sorted(active_models.keys()):
        spec = AUTHORIZED_CHECKPOINTS[k]
        arch = spec["architecture"]
        seed = spec["seed"]
        ckpt_sha = spec["sha256"]

        y_logits = np.concatenate(accumulators[k]["logits"], axis=0)
        features = np.concatenate(accumulators[k]["features"], axis=0)
        y_score = 1.0 / (1.0 + np.exp(-y_logits))

        out_path = OUTPUT_DIR / f"predictions_chexpert_frontal_{arch}_{seed}.npz"
        np.savez_compressed(
            out_path,
            y_logits=y_logits,
            y_score=y_score,
            features=features,
            raw_pneumonia=raw_pneu,
            raw_pneumonia_label=raw_pneu,
            y_true_u_zeros=y_u_zeros,
            y_true_u_ones=y_u_ones,
            mask_u_excluded=mask_u_excluded,
            mask_explicitly_labeled_only=mask_explicit_only,
            image_ids=image_ids,
            patient_ids=patient_ids,
            metadata_path=metadata_paths,
            metadata_frontal_lateral=metadata_view,
            metadata_ap_pa=metadata_ap_pa,
            metadata_sex=metadata_sex,
            metadata_age=metadata_age,
            pneumonia_col_idx=np.int64(PNEU_COL_IDX),
            label_cols=np.array(SHARED_CLASSES),
            row_order_hash=np.array(row_order_hash),
            checkpoint_sha256=np.array(ckpt_sha),
            run_id=np.array(RUN_ID),
            data_source=np.array("real_inference"),
            analysis_plan_sha256=np.array(plan_sha),
            timestamp=np.array(datetime.now(timezone.utc).isoformat()),
        )
        print(f"Saved: {out_path.name} ({out_path.stat().st_size / (1024**2):.1f} MB, SHA: {ckpt_sha[:10]}...)")
        saved_files.append(out_path)

    # Write manifest
    manifest_p = OUTPUT_DIR / "predictions_manifest.json"
    with open(manifest_p, "w") as f:
        json.dump({
            "run_id": RUN_ID,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "n_images": len(eval_df),
            "files": [f.name for f in saved_files]
        }, f, indent=2)

    print(f"\nInference completed successfully. All arrays stamped with 'real_inference'.")
    return saved_files

if CHEXPERT_DIR and not CHEXPERT_FRONTAL_DF.empty:
    SAVED_PREDS = run_single_decode_chexpert_inference(
        CHEXPERT_FRONTAL_DF, CHEXPERT_DIR, device, STAGE, SMOKE_TEST
    )
